In [1]:
!pip install pymysql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 1.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import pymysql

In [3]:
connection = pymysql.connect(
    host='18.136.157.135',
    user='dm_team5',
    password='DM!$!Team!520@4!23&',
    database='project_profit_analysis'
)

In [4]:
query = "SELECT * FROM startup"

In [5]:
df = pd.read_sql(query, connection)

/tmp/ipykernel_1396/1848774206.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


In [6]:
connection.close()

In [8]:
df.head()

,RD_Spend,Administration,Marketing_Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RD_Spend         50 non-null     float64
 1   Administration   50 non-null     float64
 2   Marketing_Spend  50 non-null     float64
 3   State            50 non-null     object 
 4   Profit           50 non-null     float64
dtypes: float64(4), object(1)
memory usage: 2.1+ KB


In [10]:
df.to_csv('startup_profit_data.csv', index=False)

In [11]:
df_encoded = pd.get_dummies(df, columns=['State'], drop_first=True)

In [12]:
df_encoded.head()

,RD_Spend,Administration,Marketing_Spend,Profit,State_Florida,State_New York
0,165349.20,136897.80,471784.10,192261.83,False,True
1,162597.70,151377.59,443898.53,191792.06,False,False
2,153441.51,101145.55,407934.54,191050.39,True,False
3,144372.41,118671.85,383199.62,182901.99,False,True
4,142107.34,91391.77,366168.42,166187.94,True,False


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [14]:
X = df_encoded.drop('Profit', axis=1)
y = df_encoded['Profit']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [19]:
new_data = {
    'RD_Spend': [21892.92, 23940.93],
    'Administration': [81910.77, 96489.63],
    'Marketing_Spend': [164270.70, 137001.10],
    'State_Florida': [0, 0],   # Neither startup is in Florida
    'State_New York': [0, 1]   # Startup 1 is California (0), Startup 2 is New York (1) as the state name was not given in documnet so i assumed first input state as California and 2nd input state as New York
}

In [20]:
df_predictions = pd.DataFrame(new_data)

In [21]:
predicted_profits = model.predict(df_predictions)

In [22]:
df_predictions['Predicted_Profit'] = predicted_profits

In [23]:
print("--- Updated Profit Predictions ---")
for index, row in df_predictions.iterrows():
    if row['State_New York'] == 1:
        state_label = "New York"
    elif row['State_Florida'] == 1:
        state_label = "Florida"
    else:
        state_label = "California"

    print(f"Startup {index + 1} ({state_label}): Estimated Profit = ${row['Predicted_Profit']:,.2f}")

--- Updated Profit Predictions ---
Startup 1 (California): Estimated Profit = $70,935.54
Startup 2 (New York): Estimated Profit = $70,775.47


In [24]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

In [25]:
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

In [27]:
r2,mse,mae,rmse

(0.8987266414328637,
 82010363.04430099,
 6961.477813252376,
 np.float64(9055.957323458464))

# PRDA-01 Profit Analysis: Final Report

## 1. Prediction Results
Using the Multiple Linear Regression model, the estimated profits for the given scenarios are:
* **Startup 1 (California):** \$70,935.54 (High Marketing, Lower R&D)
* **Startup 2 (New York):** \$70,775.47 (Lower Marketing, Higher R&D)

## 2. Model Performance Metrics
To validate the reliability of these predictions, the model was evaluated against unseen testing data:
* **R-squared (R²):** 0.8987 *(The model successfully explains 89.87% of the variance in startup profit)*
* **Mean Absolute Error (MAE):** $6,961.48 *(On average, our predictions are within ~$6.9k of the actual profit)*
* **Root Mean Squared Error (RMSE):** $9,055.96

## 3. Key Insights
* **R&D Drives Profitability:** The model indicates that Research & Development spending has a much stronger positive correlation with overall profit than Marketing spend.
* **Marketing Efficiency:** Startup 2 achieved nearly identical predicted profits to Startup 1, despite spending roughly $27,000 less on marketing. This highlights that massive marketing budgets do not automatically guarantee higher returns.

## 4. Business Recommendations
* The company should prioritize R&D funding for future startups, as it acts as the primary driver of high profit margins.
* Existing marketing budgets should be audited for efficiency, and the company should consider reallocating surplus marketing funds into R&D to maximize the return on investment.